# Qwen3 0.6B Twitter Sentiment Classification — Complete Notebook

This notebook gives you three approaches for the same binary sentiment task:

1. **Generative classification** — your current approach. The model generates `positive` or `negative`.
2. **Label-token logprob scoring** — uses the same trained LoRA model, but scores candidate labels directly instead of generating free text.
3. **Classification head** — trains a separate sequence-classification model with classifier logits.

Recommended experiment order:

- First run **Part A** to train/save the generative LoRA model.
- Then run **Part B** to evaluate the saved LoRA model using label-token logprob scoring.
- Run **Part C** only if you want a true classifier head model.

## 0. Environment check

Run this first to confirm your RTX 5090 is actually supported by your PyTorch build. For RTX 5090, the architecture list must include `sm_120`. If `sm_120` is missing, CUDA kernels may fail with `no kernel image is available for execution on the device`.

In [2]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Torch arch list:", torch.cuda.get_arch_list())

    # Simple CUDA kernel test
    x = torch.randn(4, 4, device="cuda")
    y = x @ x
    print("CUDA test passed:")
    print(y)

Python: 3.11.10 (main, Sep  7 2024, 18:35:41) [GCC 11.4.0]
Torch: 2.4.1+cu124
Torch CUDA: 12.4
CUDA available: True
GPU: NVIDIA RTX 5000 Ada Generation
Compute capability: (8, 9)
Torch arch list: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
CUDA test passed:
tensor([[ 2.4060, -0.4116, -1.0363, -2.6530],
        [ 0.5617, -1.1649, -2.4258, -0.1423],
        [-0.2727,  1.1058, -0.0543, -0.1214],
        [-0.5549,  1.0392,  1.1348,  0.2128]], device='cuda:0')


## 1. Install / update required libraries

Use this only when setting up a fresh RunPod environment. If your current environment already works with RTX 5090 and shows `sm_120`, you can skip the PyTorch install and only update Hugging Face packages if needed.

For your current working setup, the most important point is: **do not downgrade PyTorch to a wheel that does not support `sm_120`.**

In [1]:
# Optional: run only in a fresh environment if packages are missing.
# Keep this commented if your current Torch already works with RTX 5090.

!pip install -U --no-cache-dir transformers accelerate peft trl datasets scikit-learn tqdm pandas
!pip install -U hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 207.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 155.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 345.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 198.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 117.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.8/799.8 kB 179.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 117.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

## 2. Imports and global configuration

We keep a single label mapping throughout the notebook:

- `0 = negative`
- `1 = positive`

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import load_dataset, DatasetDict
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

ID2LABEL = {
    0: "negative",
    1: "positive",
}

LABEL2ID = {
    "negative": 0,
    "positive": 1,
}

LABEL_CANDIDATES = ["negative", "positive"]

SYSTEM_PROMPT = (
    "You are a strict sentiment classification model. "
    "Classify tweets as either positive or negative. "
    "Return only one word: positive or negative."
)

MODEL_NAME = "Qwen/Qwen3-0.6B"
GEN_LORA_OUTPUT_DIR = "./qwen3-0.6b-twitter-sentiment-lora"
GEN_LORA_SAVED_DIR = "./qwen3-0.6b-twitter-sentiment-lora_saved"
CLS_LORA_OUTPUT_DIR = "./qwen3-0.6b-twitter-sentiment-classifier-lora"
CLS_LORA_SAVED_DIR = "./qwen3-0.6b-twitter-sentiment-classifier-lora_saved"

DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

print("Using dtype:", DTYPE)

## 3. Load and prepare the Twitter sentiment dataset

The raw dataset files are tab-separated text files.

Each row is parsed into:

- `text`: tweet text
- `feeling`: integer label, where `0 = negative`, `1 = positive`

In [ ]:
base_url = "https://raw.githubusercontent.com/cblancac/SentimentAnalysisBert/main/data/"

data_files = {
    "train_raw": base_url + "train_150k.txt",
    "test": base_url + "test_62k.txt",
}

raw_ds = load_dataset("text", data_files=data_files)

def parse_tweet(example):
    label, text = example["text"].split("	", 1)
    return {
        "text": text.strip(),
        "feeling": int(label),
    }

parsed_ds = raw_ds.map(
    parse_tweet,
    remove_columns=["text"],
)

train_valid = parsed_ds["train_raw"].train_test_split(
    train_size=0.8,
    seed=SEED,
)

ds = DatasetDict({
    "train": train_valid["train"],
    "validation": train_valid["test"],
    "test": parsed_ds["test"],
})

print(ds)
print(ds["train"][0])
print(ds["train"].features)

## Part A — Generative classification with Qwen3 + LoRA

This is your current approach.

We convert each tweet into a chat conversation:

- system: strict sentiment classification instruction
- user: tweet + instruction
- assistant: `positive` or `negative`

The model is trained as a causal language model to generate the label word.

In [ ]:
def build_user_prompt(tweet):
    return f"""
Classify the sentiment of the following tweet.

Tweet:
{tweet}

Answer with exactly one word: positive or negative.
""".strip()


def to_chat_format(example):
    label_text = ID2LABEL[example["feeling"]]

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": build_user_prompt(example["text"]),
            },
            {
                "role": "assistant",
                "content": label_text,
            },
        ]
    }

chat_ds = ds.map(
    to_chat_format,
    remove_columns=ds["train"].column_names,
)

print(chat_ds)
print(chat_ds["train"][0])

### 4. Create a smaller training subset

For fast experimentation, train on a smaller subset first. After the pipeline works, increase `MAX_TRAIN_SAMPLES` and `MAX_VALID_SAMPLES`.

In [ ]:
MAX_TRAIN_SAMPLES = 20_000
MAX_VALID_SAMPLES = 3_000

small_chat_ds = DatasetDict({
    "train": chat_ds["train"].shuffle(seed=SEED).select(
        range(min(MAX_TRAIN_SAMPLES, len(chat_ds["train"])))
    ),
    "validation": chat_ds["validation"].shuffle(seed=SEED).select(
        range(min(MAX_VALID_SAMPLES, len(chat_ds["validation"])))
    ),
    "test": chat_ds["test"],
})

print(small_chat_ds)

### 5. Load Qwen3-0.6B as a causal language model

We use `AutoModelForCausalLM` because this part trains the model to generate the label token.

For your RTX 5090, loading directly onto GPU is fine for Qwen3-0.6B.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map={"": 0} if torch.cuda.is_available() else None,
)

model.config.use_cache = False

print("Model loaded.")
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

### 6. Configure LoRA for causal LM training

For Qwen-style decoder models, these target modules are appropriate:

- attention projections: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- MLP projections: `gate_proj`, `up_proj`, `down_proj`

In [ ]:
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

### 7. Define SFT training configuration

Important choices:

- `assistant_only_loss=True`: train only on the assistant answer, not the full prompt.
- `max_length=256`: enough for this tweet dataset. Increase if your inputs are longer.
- `bf16=True` on RTX 5090 if supported.

For higher GPU utilization, increase `per_device_train_batch_size` and/or `max_length`, but for this small model 100% utilization is not necessary.

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=GEN_LORA_OUTPUT_DIR,

    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    learning_rate=1e-4,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,

    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    tf32=True,

    max_length=256,
    packing=False,
    assistant_only_loss=True,

    logging_steps=25,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    report_to="none",
    seed=SEED,
)

### 8. Train the generative LoRA model

This trains only LoRA adapter weights, not the full Qwen model.

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=small_chat_ds["train"],
    eval_dataset=small_chat_ds["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

### 9. Save the trained LoRA adapter

This saves the PEFT adapter and tokenizer. The saved directory can be loaded later without keeping the trainer object.

In [ ]:
trainer.save_model(GEN_LORA_SAVED_DIR)
tokenizer.save_pretrained(GEN_LORA_SAVED_DIR)

print("Saved adapter to:", GEN_LORA_SAVED_DIR)
!ls -lah {GEN_LORA_SAVED_DIR}

### 10. Generative inference function

This function uses `model.generate()` to generate the label. This is simple, but less controlled than label-token scoring because the model can generate extra text.

In [ ]:
def normalize_generated_label(response):
    response = response.lower().strip()

    if "positive" in response:
        return "positive"
    if "negative" in response:
        return "negative"
    return "unknown"


def build_chat_prompt_for_inference(tweet, tokenizer_obj):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_user_prompt(tweet),
        },
    ]

    try:
        return tokenizer_obj.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer_obj.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def get_model_device(model_obj):
    return next(model_obj.parameters()).device


@torch.no_grad()
def classify_tweet_generate(model_obj, tokenizer_obj, tweet, max_new_tokens=5):
    model_obj.eval()

    prompt = build_chat_prompt_for_inference(tweet, tokenizer_obj)
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(get_model_device(model_obj))

    outputs = model_obj.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer_obj.eos_token_id,
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_response = tokenizer_obj.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    prediction = normalize_generated_label(raw_response)

    return {
        "tweet": tweet,
        "raw_response": raw_response,
        "prediction": prediction,
    }

### 11. Test generative inference on one validation row

Change `split_name` and `row_index` to inspect any example from validation or test.

In [ ]:
split_name = "validation"   # choose: "train", "validation", or "test"
row_index = 0               # change this index

row = ds[split_name][row_index]
tweet = row["text"]
true_label = ID2LABEL[row["feeling"]]

result = classify_tweet_generate(trainer.model, tokenizer, tweet)

print("Tweet:", tweet)
print("True label:", true_label)
print("Raw model response:", result["raw_response"])
print("Prediction:", result["prediction"])
print("Correct:", result["prediction"] == true_label)

### 12. Evaluate generative inference on a sample of the test set

This uses `generate()`. It is useful as a baseline, but it can be slower and less deterministic than label-token logprob scoring.

In [ ]:
def evaluate_generate(model_obj, tokenizer_obj, split="test", sample_size=1000):
    eval_sample = ds[split].shuffle(seed=SEED)

    if sample_size is not None:
        eval_sample = eval_sample.select(range(min(sample_size, len(eval_sample))))

    preds = []
    labels = []
    records = []

    for row in tqdm(eval_sample):
        tweet = row["text"]
        true_id = row["feeling"]
        true_label = ID2LABEL[true_id]

        result = classify_tweet_generate(model_obj, tokenizer_obj, tweet)
        pred_label = result["prediction"]
        pred_id = LABEL2ID.get(pred_label, -1)

        preds.append(pred_id)
        labels.append(true_id)

        records.append({
            "text": tweet,
            "true_label": true_label,
            "raw_response": result["raw_response"],
            "pred_label": pred_label,
            "correct": pred_id == true_id,
        })

    valid_pairs = [(y, p) for y, p in zip(labels, preds) if p in [0, 1]]
    valid_labels = [y for y, p in valid_pairs]
    valid_preds = [p for y, p in valid_pairs]

    print("Valid predictions:", len(valid_preds), "/", len(preds))
    print("Accuracy:", accuracy_score(valid_labels, valid_preds))
    print(classification_report(valid_labels, valid_preds, target_names=["negative", "positive"]))
    print("Confusion matrix:")
    print(confusion_matrix(valid_labels, valid_preds))

    return pd.DataFrame(records)

# Uncomment to run generative evaluation
# gen_results_df = evaluate_generate(trainer.model, tokenizer, split="test", sample_size=1000)
# gen_results_df.head()

## Part B — Load saved LoRA model and use label-token logprob scoring

This is the recommended upgrade to your current notebook.

Instead of generating text, we score the candidate labels directly:

- `P("negative" | prompt)`
- `P("positive" | prompt)`

Then we choose the label with the higher average log probability.

Benefits:

- no free-form output
- deterministic
- cleaner evaluation
- works with the LoRA model you already trained
- no retraining needed

### 13. Free memory before reloading the saved model

This is optional, but useful if you trained in the same notebook session.

In [ ]:
# Optional memory cleanup before loading saved model separately
try:
    del trainer
except NameError:
    pass

try:
    del model
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 14. Load the saved PEFT/LoRA model from disk

This is the correct way to load the adapter after training. It does not require the original `trainer` object.

In [ ]:
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

adapter_dir = GEN_LORA_SAVED_DIR

loaded_tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

if loaded_tokenizer.pad_token is None:
    loaded_tokenizer.pad_token = loaded_tokenizer.eos_token

loaded_model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_dir,
    dtype=DTYPE,
    device_map={"": 0} if torch.cuda.is_available() else None,
)

loaded_model.eval()

print("Loaded saved LoRA model from:", adapter_dir)
print("Device:", get_model_device(loaded_model))
print("Active adapter:", loaded_model.active_adapter)

### 15. Label-token logprob scoring functions

This computes how likely each label is after the prompt.

We use `avg_logprob` rather than total log probability to avoid bias against labels that tokenize into more tokens.

In [ ]:
@torch.no_grad()
def score_label_given_prompt(model_obj, tokenizer_obj, prompt, label):
    """
    Compute log probability of a candidate label after the prompt.
    Example: score P("positive" | prompt) and P("negative" | prompt).
    """
    model_obj.eval()
    device = get_model_device(model_obj)

    prompt_ids = tokenizer_obj(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(device)

    label_ids = tokenizer_obj(
        label,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(device)

    input_ids = torch.cat([prompt_ids, label_ids], dim=1)
    attention_mask = torch.ones_like(input_ids).to(device)

    outputs = model_obj(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

    logits = outputs.logits

    prompt_len = prompt_ids.shape[1]
    label_len = label_ids.shape[1]

    # Token t is predicted by logits from the previous position.
    label_logits = logits[0, prompt_len - 1 : prompt_len + label_len - 1, :]
    log_probs = F.log_softmax(label_logits.float(), dim=-1)

    selected_log_probs = log_probs.gather(
        dim=-1,
        index=label_ids[0].unsqueeze(-1),
    ).squeeze(-1)

    total_logprob = selected_log_probs.sum().item()
    avg_logprob = selected_log_probs.mean().item()

    return {
        "label": label,
        "total_logprob": total_logprob,
        "avg_logprob": avg_logprob,
        "num_tokens": label_len,
        "token_ids": label_ids[0].tolist(),
    }


def classify_tweet_by_label_logprob(model_obj, tokenizer_obj, tweet):
    prompt = build_chat_prompt_for_inference(tweet, tokenizer_obj)

    scores = {}
    for label in LABEL_CANDIDATES:
        scores[label] = score_label_given_prompt(
            model_obj=model_obj,
            tokenizer_obj=tokenizer_obj,
            prompt=prompt,
            label=label,
        )

    best_label = max(
        scores.keys(),
        key=lambda label: scores[label]["avg_logprob"],
    )

    raw_scores = torch.tensor(
        [scores[label]["avg_logprob"] for label in LABEL_CANDIDATES],
        dtype=torch.float32,
    )

    probs = torch.softmax(raw_scores, dim=0).tolist()
    label_probs = {
        label: prob
        for label, prob in zip(LABEL_CANDIDATES, probs)
    }

    return {
        "tweet": tweet,
        "prediction": best_label,
        "scores": scores,
        "probabilities": label_probs,
    }

### 16. Test label-token scoring on one validation or test row

This is the cell you should use when you want to load a particular text from validation/test and check the model prediction.

In [ ]:
split_name = "validation"  # choose: "train", "validation", or "test"
row_index = 0              # change this to inspect another example

row = ds[split_name][row_index]
tweet = row["text"]
true_label = ID2LABEL[row["feeling"]]

result = classify_tweet_by_label_logprob(
    loaded_model,
    loaded_tokenizer,
    tweet,
)

print("Tweet:", tweet)
print("True label:", true_label)
print("Prediction:", result["prediction"])
print("Probabilities:", result["probabilities"])
print("Scores:", result["scores"])
print("Correct:", result["prediction"] == true_label)

### 17. Predict any custom text

Use this cell when you want to type your own tweet/review.

In [ ]:
custom_text = "Nice one. Bluetooth calling is great, but step tracking is not accurate."

result = classify_tweet_by_label_logprob(
    loaded_model,
    loaded_tokenizer,
    custom_text,
)

print("Text:", custom_text)
print("Prediction:", result["prediction"])
print("Probabilities:", result["probabilities"])
print("Scores:", result["scores"])

### 18. Evaluate validation or test set using label-token logprob scoring

This is usually better than `generate()` for classification because it forces the decision among the allowed labels.

In [ ]:
def evaluate_label_logprob(model_obj, tokenizer_obj, split="test", sample_size=1000):
    eval_sample = ds[split].shuffle(seed=SEED)

    if sample_size is not None:
        eval_sample = eval_sample.select(range(min(sample_size, len(eval_sample))))

    preds = []
    labels = []
    records = []

    for row in tqdm(eval_sample):
        tweet = row["text"]
        true_id = row["feeling"]
        true_label = ID2LABEL[true_id]

        result = classify_tweet_by_label_logprob(
            model_obj,
            tokenizer_obj,
            tweet,
        )

        pred_label = result["prediction"]
        pred_id = LABEL2ID[pred_label]

        preds.append(pred_id)
        labels.append(true_id)

        records.append({
            "text": tweet,
            "true_label": true_label,
            "pred_label": pred_label,
            "negative_prob": result["probabilities"]["negative"],
            "positive_prob": result["probabilities"]["positive"],
            "negative_avg_logprob": result["scores"]["negative"]["avg_logprob"],
            "positive_avg_logprob": result["scores"]["positive"]["avg_logprob"],
            "correct": pred_id == true_id,
        })

    print("Accuracy:", accuracy_score(labels, preds))
    print(classification_report(labels, preds, target_names=["negative", "positive"]))
    print("Confusion matrix:")
    print(confusion_matrix(labels, preds))

    return pd.DataFrame(records)

logprob_results_df = evaluate_label_logprob(
    loaded_model,
    loaded_tokenizer,
    split="test",
    sample_size=1000,
)

logprob_results_df.head()

### 19. Inspect mistakes

This helps you understand where the fine-tuned model is confused.

In [ ]:
mistakes_df = logprob_results_df[logprob_results_df["correct"] == False].copy()

print("Number of mistakes:", len(mistakes_df))
mistakes_df.head(20)

## Part C — Optional classification-head model

This is a separate training path.

Use this if you want a true classifier with logits/probabilities instead of a generative model.

Important:

- Do **not** reuse the causal-LM LoRA adapter directly.
- Train a separate model using `AutoModelForSequenceClassification`.
- Use `TaskType.SEQ_CLS` in PEFT.
- Save this as a separate adapter directory.

### 20. Free memory before classification-head training

This prevents GPU memory issues if you ran the generative model in the same session.

In [ ]:
try:
    del loaded_model
except NameError:
    pass

try:
    del loaded_tokenizer
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 21. Load Qwen3 with a sequence-classification head

This changes the model from:

`tweet → next-token generation`

to:

`tweet → classifier logits [negative, positive]`

In [3]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from peft import get_peft_model

clf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if clf_tokenizer.pad_token is None:
    clf_tokenizer.pad_token = clf_tokenizer.eos_token

clf_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    dtype=DTYPE,
)

clf_base_model.config.pad_token_id = clf_tokenizer.pad_token_id
clf_base_model.config.problem_type = "single_label_classification"
clf_base_model.config.use_cache = False

print("Classification model loaded.")
print("Classifier-related modules:")
for name, module in clf_base_model.named_modules():
    if "score" in name or "classifier" in name:
        print(name, module)

NameError: name 'AutoTokenizer' is not defined

### 22. Add LoRA for sequence classification

`modules_to_save=["score"]` keeps the classification head trainable and saved with the adapter. For Qwen sequence classification, the classifier head is usually named `score`.

In [ ]:
clf_peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=["score"],
)

clf_model = get_peft_model(clf_base_model, clf_peft_config)
clf_model.print_trainable_parameters()

### 23. Tokenize dataset for classifier-head training

Unlike SFT, this uses raw tweets and numeric labels.

Input:

- `text`

Target:

- `labels`

In [ ]:
def tokenize_for_classification(batch):
    return clf_tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )

clf_ds = ds.map(
    tokenize_for_classification,
    batched=True,
)

clf_ds = clf_ds.rename_column("feeling", "labels")

# Keep only model input columns plus labels
columns_to_keep = {"input_ids", "attention_mask", "labels"}
columns_to_remove = [
    col for col in clf_ds["train"].column_names
    if col not in columns_to_keep
]

clf_ds = clf_ds.remove_columns(columns_to_remove)

print(clf_ds)
print(clf_ds["train"][0])

### 24. Classification-head metrics

For binary sentiment classification, we track accuracy and F1.

In [ ]:
def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": classification_report(
            labels,
            preds,
            target_names=["negative", "positive"],
            output_dict=True,
            zero_division=0,
        )["weighted avg"]["f1-score"],
    }

data_collator = DataCollatorWithPadding(tokenizer=clf_tokenizer)

### 25. Train the classification-head LoRA model

This trains LoRA adapters plus the classification head.

In [ ]:
clf_training_args = TrainingArguments(
    output_dir=CLS_LORA_OUTPUT_DIR,

    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,

    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    tf32=True,

    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    logging_steps=25,
    report_to="none",
    seed=SEED,
)

clf_trainer = Trainer(
    model=clf_model,
    args=clf_training_args,
    train_dataset=clf_ds["train"],
    eval_dataset=clf_ds["validation"],
    tokenizer=clf_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_cls_metrics,
)

clf_trainer.train()

### 26. Evaluate the classification-head model on test set

This gives direct classifier performance.

In [ ]:
clf_test_metrics = clf_trainer.evaluate(clf_ds["test"])
print(clf_test_metrics)

### 27. Save the classification-head LoRA model

In [ ]:
clf_trainer.save_model(CLS_LORA_SAVED_DIR)
clf_tokenizer.save_pretrained(CLS_LORA_SAVED_DIR)

print("Saved classifier adapter to:", CLS_LORA_SAVED_DIR)
!ls -lah {CLS_LORA_SAVED_DIR}

### 28. Classification-head inference function

This returns direct softmax probabilities over `negative` and `positive`.

In [ ]:
@torch.no_grad()
def classify_tweet_with_head(model_obj, tokenizer_obj, tweet):
    model_obj.eval()
    device = get_model_device(model_obj)

    inputs = tokenizer_obj(
        tweet,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(device)

    outputs = model_obj(**inputs)
    logits = outputs.logits[0]
    probs = F.softmax(logits.float(), dim=-1)

    pred_id = torch.argmax(probs).item()
    pred_label = ID2LABEL[pred_id]

    return {
        "tweet": tweet,
        "prediction": pred_label,
        "probabilities": {
            ID2LABEL[i]: probs[i].item()
            for i in range(len(ID2LABEL))
        },
    }

custom_text = "I love this product. It works perfectly."
result = classify_tweet_with_head(clf_model, clf_tokenizer, custom_text)
print(result)

## Final recommendation

Use all three results for comparison:

1. **Generative classification** — easiest and closest to chat-style SFT.
2. **Label-token logprob scoring** — best upgrade for your current LoRA model; no retraining required.
3. **Classification head** — best if the final task is only sentiment classification and you want direct logits/probabilities.

For your current notebook and saved LoRA adapter, the most practical next step is:

```text
Use Part B: label-token logprob scoring on the saved generative LoRA model.
```

Then compare it against the classification-head model if you train Part C.